In [8]:


# تثبيت الحزم الخاصة بالعرض التسجيلي في Colab
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
!pip install gymnasium pygame moviepy imageio imageio-ffmpeg > /dev/null 2>&1

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML
from base64 import b64encode

# إعداد الشاشة الافتراضية لتسجيل الفيديو
os.environ["DISPLAY"] = ":99"
!Xvfb :99 -screen 0 1024x768x24 > /dev/null 2>&1 &

In [9]:
# --------------------------------------------------
# 1. بناء الشبكة العصبية (Q-Network)
# --------------------------------------------------
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        return self.fc(x)

# --------------------------------------------------
# 2. ذاكرة إعادة التجربة (Replay Buffer)
# --------------------------------------------------
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        return (torch.FloatTensor(np.array(state)),
                torch.LongTensor(action),
                torch.FloatTensor(reward),
                torch.FloatTensor(np.array(next_state)),
                torch.FloatTensor(done))

    def __len__(self):
        return len(self.buffer)

# --------------------------------------------------
# 3. إعداد البيئة والبرامترات الفائقة
# --------------------------------------------------
env_name = "CartPole-v1"
env = gym.make(env_name)

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# Hyperparameters
gamma = 0.99
lr = 0.001
batch_size = 64
capacity = 10000
epsilon_start = 1.0
epsilon_end = 0.01
epsilon_decay = 0.995
tau = 0.005 # Soft update parameter

policy_net = DQN(state_dim, action_dim)
target_net = DQN(state_dim, action_dim)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=lr)
buffer = ReplayBuffer(capacity)

# --------------------------------------------------
# 4. تسجيل فيديو لأداء العميل العشوائي (قبل التدريب)
# --------------------------------------------------
print("📹 تسجيل فيديو للأداء العشوائي قبل التدريب...")
untrained_env = gym.make(env_name, render_mode="rgb_array")
untrained_env = RecordVideo(untrained_env, video_folder="./video_untrained", episode_trigger=lambda x: True)

state, _ = untrained_env.reset()
done = False
while not done:
    action = untrained_env.action_space.sample() # حركة عشوائية
    state, reward, terminated, truncated, _ = untrained_env.step(action)
    done = terminated or truncated
untrained_env.close()

# --------------------------------------------------
# 5. حلقة التدريب (Training Loop)
# --------------------------------------------------
episodes = 200
epsilon = epsilon_start

print("\n🚀 بدء تدريب شبكة DQN...")
for episode in range(episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        # سياسة Epsilon-Greedy
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                st_tensor = torch.FloatTensor(state).unsqueeze(0)
                action = policy_net(st_tensor).argmax().item()

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        buffer.push(state, action, reward, next_state, float(done))
        state = next_state
        total_reward += reward

        # التحديث من الذاكرة
        if len(buffer) >= batch_size:
            b_state, b_action, b_reward, b_next_state, b_done = buffer.sample(batch_size)

            q_values = policy_net(b_state)
            q_value = q_values.gather(1, b_action.unsqueeze(1)).squeeze(1)

            with torch.no_grad():
                max_next_q = target_net(b_next_state).max(1)[0]
                target_q = b_reward + (1 - b_done) * gamma * max_next_q

            loss = nn.MSELoss()(q_value, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Soft Update لشبكة Target
            for target_param, policy_param in zip(target_net.parameters(), policy_net.parameters()):
                target_param.data.copy_(tau * policy_param.data + (1.0 - tau) * target_param.data)

    epsilon = max(epsilon_end, epsilon * epsilon_decay)

    if (episode + 1) % 20 == 0:
        print(f"Episode {episode + 1}/{episodes} | Total Reward: {total_reward:.1f} | Epsilon: {epsilon:.2f}")

env.close()

# --------------------------------------------------
# 6. التقييم وتسجيل فيديو للنموذج المدرب
# --------------------------------------------------
print("\n🎬 تسجيل فيديو لأداء النموذج المدرب...")
eval_env = gym.make(env_name, render_mode="rgb_array")
eval_env = RecordVideo(eval_env, video_folder="./video_trained", episode_trigger=lambda x: True)

state, _ = eval_env.reset()
eval_reward = 0
done = False

policy_net.eval()
while not done:
    with torch.no_grad():
        st_tensor = torch.FloatTensor(state).unsqueeze(0)
        action = policy_net(st_tensor).argmax().item()

    state, reward, terminated, truncated, _ = eval_env.step(action)
    eval_reward += reward
    done = terminated or truncated

eval_env.close()
print(f"✅ نتيجة تقييم النموذج المدرب (Total Reward): {eval_reward}")

📹 تسجيل فيديو للأداء العشوائي قبل التدريب...


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"



🚀 بدء تدريب شبكة DQN...
Episode 20/200 | Total Reward: 23.0 | Epsilon: 0.90
Episode 40/200 | Total Reward: 27.0 | Epsilon: 0.82
Episode 60/200 | Total Reward: 25.0 | Epsilon: 0.74
Episode 80/200 | Total Reward: 36.0 | Epsilon: 0.67
Episode 100/200 | Total Reward: 54.0 | Epsilon: 0.61
Episode 120/200 | Total Reward: 35.0 | Epsilon: 0.55
Episode 140/200 | Total Reward: 152.0 | Epsilon: 0.50
Episode 160/200 | Total Reward: 22.0 | Epsilon: 0.45
Episode 180/200 | Total Reward: 24.0 | Epsilon: 0.41
Episode 200/200 | Total Reward: 20.0 | Epsilon: 0.37

🎬 تسجيل فيديو لأداء النموذج المدرب...
✅ نتيجة تقييم النموذج المدرب (Total Reward): 102.0


In [10]:
def show_video(video_folder):
    files = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]
    if not files:
        print("لم يتم العثور على فيديو!")
        return
    video_path = os.path.join(video_folder, files[0])
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <video width=400 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)

print("--- فيديو العميل قبل التدريب (العشوائي) ---")
display(show_video('./video_untrained'))

print("\n--- فيديو العميل بعد التدريب (DQN) ---")
display(show_video('./video_trained'))

--- فيديو العميل قبل التدريب (العشوائي) ---



--- فيديو العميل بعد التدريب (DQN) ---
